In [ ]:
import torch
import numpy as np
import pandas as pd
from BioClinicalBERTEmbeddings import BioClinicalBERTEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
def get_patient_embedding(patient_notes, embedder):
    """
    Args:
        patient_notes: A list of strings, where each string is ONE clinical note.
        embedder: Instance of BioClinicalBERTEmbeddings
    """
    # 1. Get embeddings for ALL notes separately
    # Result shape: (num_notes, 768)
    note_embeddings = embedder.embed_documents(patient_notes)

    # Convert to tensor for easier math
    note_tensor = torch.tensor(note_embeddings)

    # 2. Aggregate into one Patient Vector
    # Average/max across the 'num_notes' dimension (dim 0)
    patient_vector = torch.max(note_tensor, dim=0).values

    return patient_vector.cpu().numpy()

In [ ]:
def add_embedding_each_patient(embed_model="bioclinicalbert"):
    """
    Each empi will have one corresponding embedding that represents all the notes for that patient,
    """
    df = pd.read_csv(f"../data/merged_data.csv")
    labels = pd.read_csv("../data/corrected_everything.csv")
    empis = []
    embeddings = []
    if embed_model == "bioclinicalbert":
        embedder = BioClinicalBERTEmbeddings()
    else:
        print(f"Using huggingfaceembeddings library for embed model: {embed_model}")
        embedder = HuggingFaceEmbeddings(
            model_name=embed_model,
            model_kwargs={'device': 'cuda'}, # Uses GPU automatically if available
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True}
        )

    for empi in df['empi'].unique():
        empis.append(empi)
        empi_notes = df[df['empi'] == empi]['clinical_notes'].to_list()
        print(f"Patient {empi} has {len(empi_notes)} notes, total of {sum(len(note.split()) for note in empi_notes)} words")

        empi_embedding = get_patient_embedding(empi_notes, embedder)
        embeddings.append(empi_embedding)

    embedded_df = pd.DataFrame({'empi': empis, 'embedding': embeddings})
    embedded_merged = pd.merge(embedded_df, labels, on='empi')
    embedded_merged.to_pickle(f"../data/merged_data_embedded_{embed_model.replace("/", "")}_maxpool.pkl")

In [ ]:
def add_embedding_each_patient_test(embed_model="bioclinicalbert"):
    """
    Each empi will have one corresponding embedding that represents all the notes for that patient in the test dataset
    """
    df = pd.read_csv(f"../data/test_merged_data.csv")
    labels = pd.read_csv("../data/test_dataset_label.csv")
    empis = []
    embeddings = []

    if embed_model == "bioclinicalbert":
            embedder = BioClinicalBERTEmbeddings()
    else:
        print(f"Using huggingfaceembeddings library for embed model: {embed_model}")
        embedder = HuggingFaceEmbeddings(
            model_name=embed_model,
            model_kwargs={'device': 'cuda'}, # Uses GPU automatically if available
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True}
        )

    for empi in df['empi'].unique():
        empis.append(empi)
        empi_notes = df[df['empi'] == empi]['clinical_notes'].to_list()
        print(f"Patient {empi} has {len(empi_notes)} notes, total of {sum(len(note.split()) for note in empi_notes)} words")

        empi_embedding = get_patient_embedding(empi_notes, embedder)
        embeddings.append(empi_embedding)

    embedded_df = pd.DataFrame({'empi': empis, 'embedding': embeddings})
    embedded_merged = pd.merge(embedded_df, labels, on='empi')
    embedded_merged.to_pickle(f"../data/test_merged_data_embedded_{embed_model.replace("/", "")}_maxpool.pkl")

In [ ]:
add_embedding_each_patient("pritamdeka/S-PubMedBert-MS-MARCO")

In [ ]:
add_embedding_each_patient_test("pritamdeka/S-PubMedBert-MS-MARCO")